# Momentum Strategy — Assignment-Rubric Visualizations
Maps directly to the 4-step brief: *"Does your strategy have real Alpha?"*

| Step | Requirement | Chart(s) in this notebook |
|---|---|---|
| 3 | Backtest via rolling window — return, volatility, Sharpe, max drawdown vs. buy-and-hold | Equity Curve, Rolling Metrics Panel, Drawdown Comparison, Summary Metrics Bar Chart |
| 4 | Regress excess returns on the market — is α positive & significant? | CAPM Scatter & Fit, Rolling Alpha |

Reads the same processed parquet files as the pipeline scripts:
- `momentum_strategy_returns.parquet`
- `backtest_master_panel.parquet`


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import statsmodels.api as sm

plt.rcParams['figure.dpi'] = 110
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

RF_MONTHLY = 0.003   # matches 04_evaluate_performance.py
ANN_FACTOR = 12
ROLL_WINDOW = 24      # rolling window for return/vol/Sharpe/alpha/beta panels (months)

processed_dir = '../data/processed' if os.path.exists('../data/processed') else '.'
PORTFOLIO_PATH = os.path.join(processed_dir, "momentum_strategy_returns.parquet")
MASTER_PANEL_PATH = os.path.join(processed_dir, "backtest_master_panel.parquet")

## Load Data & Rebuild the Market Proxy

Same logic as `04_evaluate_performance.py` (Step 5) so every chart here ties back to the numbers already reported: strategy −24.83% vol / −0.11 Sharpe vs. market 10.20% return / 0.37 Sharpe.

In [ ]:
strat_df = pd.read_parquet(PORTFOLIO_PATH)
panel_df = pd.read_parquet(MASTER_PANEL_PATH)

strat_df['rebalance_date'] = pd.to_datetime(strat_df['rebalance_date'])
panel_df['rebalance_date'] = pd.to_datetime(panel_df['rebalance_date'])

panel_df = panel_df.sort_values(['permno', 'rebalance_date'])
panel_df['next_date'] = panel_df.groupby('permno')['rebalance_date'].shift(-1)
panel_df['holding_return_t1'] = panel_df.groupby('permno')['total_return'].shift(-1)

invalid_gap = (panel_df['next_date'] - panel_df['rebalance_date']).dt.days > 35
panel_df.loc[invalid_gap, 'holding_return_t1'] = np.nan

market_df = panel_df.groupby('rebalance_date')['holding_return_t1'].mean().reset_index()
market_df.rename(columns={'holding_return_t1': 'Market_Return'}, inplace=True)

df = pd.merge(strat_df, market_df, on='rebalance_date').dropna(subset=['Market_Return'])
df = df.sort_values('rebalance_date').reset_index(drop=True)

df['Realized_Spread'] = df['Long_Leg'] - df['Short_Leg']
df['Strategy_Excess'] = df['Long_Short_Spread']
df['Market_Excess'] = df['Market_Return'] - RF_MONTHLY

print(f"Merged panel: {len(df)} months, {df['rebalance_date'].min().date()} to {df['rebalance_date'].max().date()}")

---
## STEP 3 — Backtest via Rolling Window
### 1. Equity Curve: Strategy vs. Buy-and-Hold Benchmark

Foregrounds the strategy-vs-benchmark comparison the assignment asks for. `Long_Short_Spread` is a zero-cost overlay (spread P&L growth), while `Market_Return` represents a fully-funded buy-and-hold position.

In [ ]:
cum = pd.DataFrame({'rebalance_date': df['rebalance_date']})
cum['Long_Short_Spread'] = (1 + df['Long_Short_Spread']).cumprod()
cum['Market_Return'] = (1 + df['Market_Return']).cumprod()

fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(cum['rebalance_date'], cum['Market_Return'], color='#1f77b4', lw=2.3,
        label='Buy-and-Hold Benchmark (Equal-Weighted Market)')
ax.plot(cum['rebalance_date'], cum['Long_Short_Spread'], color='#7f7f7f', lw=2.3, ls='--',
        label='Momentum Long-Short Spread')

ax.axhline(1.0, color='black', lw=0.8, alpha=0.5)
ax.set_yscale('log')
ax.set_ylabel('Growth of $1 (log scale)')
ax.set_xlabel('Date')
ax.set_title('Strategy vs. Buy-and-Hold Benchmark: Cumulative Returns\n2000–2024, Monthly Rebalanced')
ax.legend(loc='upper left', frameon=False)
ax.xaxis.set_major_locator(mdates.YearLocator(2))
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
fig.autofmt_xdate()
plt.tight_layout()
plt.savefig('step3_equity_curve_vs_benchmark.png', dpi=150, bbox_inches='tight')
plt.show()

### 2. Rolling Window Metrics Panel (Return, Volatility, Sharpe)

The brief lists "backtest it via rolling window" as its own step, distinct from a single full-sample point estimate. This computes 24-month rolling annualized return, volatility, and Sharpe for both the strategy and the benchmark, stacked on a shared x-axis.

In [ ]:
def rolling_ann_return(x):
    return x.mean() * ANN_FACTOR

def rolling_ann_vol(x):
    return x.std() * np.sqrt(ANN_FACTOR)

roll = pd.DataFrame({'rebalance_date': df['rebalance_date']})
roll['strat_ret'] = df['Long_Short_Spread'].rolling(ROLL_WINDOW).apply(rolling_ann_return, raw=False)
roll['strat_vol'] = df['Long_Short_Spread'].rolling(ROLL_WINDOW).apply(rolling_ann_vol, raw=False)
roll['strat_sharpe'] = roll['strat_ret'] / roll['strat_vol']

roll['mkt_ret'] = df['Market_Return'].rolling(ROLL_WINDOW).apply(rolling_ann_return, raw=False)
roll['mkt_vol'] = df['Market_Return'].rolling(ROLL_WINDOW).apply(rolling_ann_vol, raw=False)
roll['mkt_excess_ret'] = roll['mkt_ret'] - RF_MONTHLY * ANN_FACTOR
roll['mkt_sharpe'] = roll['mkt_excess_ret'] / roll['mkt_vol']

roll = roll.dropna().reset_index(drop=True)

fig, axes = plt.subplots(3, 1, figsize=(12, 10), sharex=True)

axes[0].plot(roll['rebalance_date'], roll['strat_ret'] * 100, color='#7f7f7f', lw=1.8, label='Strategy (Spread)')
axes[0].plot(roll['rebalance_date'], roll['mkt_ret'] * 100, color='#1f77b4', lw=1.8, label='Benchmark (Market)')
axes[0].axhline(0, color='black', lw=0.6, alpha=0.5)
axes[0].set_ylabel('Ann. Return (%)')
axes[0].set_title(f'Rolling {ROLL_WINDOW}-Month Annualized Return, Volatility & Sharpe: Strategy vs. Benchmark')
axes[0].legend(loc='upper left', frameon=False, fontsize=9)

axes[1].plot(roll['rebalance_date'], roll['strat_vol'] * 100, color='#7f7f7f', lw=1.8)
axes[1].plot(roll['rebalance_date'], roll['mkt_vol'] * 100, color='#1f77b4', lw=1.8)
axes[1].set_ylabel('Ann. Volatility (%)')

axes[2].plot(roll['rebalance_date'], roll['strat_sharpe'], color='#7f7f7f', lw=1.8)
axes[2].plot(roll['rebalance_date'], roll['mkt_sharpe'], color='#1f77b4', lw=1.8)
axes[2].axhline(0, color='black', lw=0.6, alpha=0.5)
axes[2].set_ylabel('Sharpe Ratio')
axes[2].set_xlabel('Date')
axes[2].xaxis.set_major_locator(mdates.YearLocator(2))
axes[2].xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
fig.autofmt_xdate()

plt.tight_layout()
plt.savefig('step3_rolling_metrics_panel.png', dpi=150, bbox_inches='tight')
plt.show()

### 3. Max Drawdown: Strategy vs. Benchmark

Extends the spread-only drawdown chart to include the benchmark's drawdown on the same axes, since Step 3 asks for max drawdown *against* the benchmark, not standalone.

In [ ]:
def compute_drawdown(returns):
    cum_wealth = (1 + returns).cumprod()
    running_max = cum_wealth.cummax()
    dd = cum_wealth / running_max - 1
    return dd

strat_dd = compute_drawdown(df['Long_Short_Spread'])
mkt_dd = compute_drawdown(df['Market_Return'])

strat_max_dd = strat_dd.min()
strat_max_dd_date = df['rebalance_date'].iloc[strat_dd.values.argmin()]
mkt_max_dd = mkt_dd.min()
mkt_max_dd_date = df['rebalance_date'].iloc[mkt_dd.values.argmin()]

fig, ax = plt.subplots(figsize=(12, 5.5))
ax.fill_between(df['rebalance_date'], strat_dd.values * 100, 0, color='#7f7f7f', alpha=0.30, label='Strategy Drawdown')
ax.plot(df['rebalance_date'], strat_dd.values * 100, color='#4d4d4d', lw=1)

ax.fill_between(df['rebalance_date'], mkt_dd.values * 100, 0, color='#1f77b4', alpha=0.25, label='Benchmark Drawdown')
ax.plot(df['rebalance_date'], mkt_dd.values * 100, color='#1f77b4', lw=1)

ax.axhline(0, color='black', lw=0.8)
ax.set_ylabel('Drawdown (%)')
ax.set_xlabel('Date')
ax.set_title('Peak-to-Trough Drawdowns: Strategy vs. Benchmark')
ax.legend(loc='lower left', frameon=False)
ax.xaxis.set_major_locator(mdates.YearLocator(2))
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
fig.autofmt_xdate()
plt.tight_layout()
plt.savefig('step3_drawdown_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Strategy max drawdown:  {strat_max_dd:.2%} ({strat_max_dd_date.date()})")
print(f"Benchmark max drawdown: {mkt_max_dd:.2%} ({mkt_max_dd_date.date()})")

### 4. Summary Metrics Bar Chart (Return / Volatility / Sharpe / Max Drawdown)

A single scannable chart confirming all four Step-3 metrics are reported for both strategy and benchmark side by side.

In [ ]:
strat_ann_ret = df['Long_Short_Spread'].mean() * ANN_FACTOR
strat_ann_vol = df['Long_Short_Spread'].std() * np.sqrt(ANN_FACTOR)
strat_sharpe = strat_ann_ret / strat_ann_vol

mkt_ann_ret = (1 + df['Market_Return']).prod() ** (ANN_FACTOR / len(df)) - 1
mkt_ann_vol = df['Market_Return'].std() * np.sqrt(ANN_FACTOR)
mkt_sharpe = (mkt_ann_ret - RF_MONTHLY * ANN_FACTOR) / mkt_ann_vol

metrics = ['Ann. Return', 'Ann. Volatility', 'Sharpe Ratio', 'Max Drawdown']
strat_vals = [strat_ann_ret * 100, strat_ann_vol * 100, strat_sharpe, strat_max_dd * 100]
mkt_vals = [mkt_ann_ret * 100, mkt_ann_vol * 100, mkt_sharpe, mkt_max_dd * 100]

fig, axes = plt.subplots(1, 4, figsize=(14, 4.5))
for i, (ax, metric) in enumerate(zip(axes, metrics)):
    bars = ax.bar(['Strategy', 'Benchmark'], [strat_vals[i], mkt_vals[i]],
                   color=['#7f7f7f', '#1f77b4'], edgecolor='black', linewidth=0.6, width=0.55)
    ax.axhline(0, color='black', lw=0.7)
    ax.set_title(metric, fontsize=10.5)
    for bar, val in zip(bars, [strat_vals[i], mkt_vals[i]]):
        suffix = '' if metric == 'Sharpe Ratio' else '%'
        ax.annotate(f'{val:.2f}{suffix}', xy=(bar.get_x() + bar.get_width()/2, val),
                    xytext=(0, 3 if val >= 0 else -12), textcoords='offset points',
                    ha='center', fontsize=9)

fig.suptitle('Step 3 Summary: Strategy vs. Benchmark — Return, Volatility, Sharpe & Max Drawdown', fontsize=12.5, y=1.03)
plt.tight_layout()
plt.savefig('step3_summary_metrics_bar.png', dpi=150, bbox_inches='tight')
plt.show()

---
## STEP 4 — Regress Excess Returns on the Market
### 5. CAPM Regression Scatter & Fit Line

Direct visual answer to "is α positive & significant?" — the annotated t-stat and p-value make the significance test explicit on the chart itself.

In [ ]:
X = sm.add_constant(df['Market_Excess'])
y = df['Strategy_Excess']
model = sm.OLS(y, X).fit()

alpha_monthly = model.params['const']
alpha_annualized = alpha_monthly * ANN_FACTOR
beta = model.params['Market_Excess']
t_stat_alpha = model.tvalues['const']
p_value_alpha = model.pvalues['const']

x_range = np.linspace(df['Market_Excess'].min(), df['Market_Excess'].max(), 100)
y_fit = alpha_monthly + beta * x_range

fig, ax = plt.subplots(figsize=(9, 7))
ax.scatter(df['Market_Excess'] * 100, df['Strategy_Excess'] * 100,
           alpha=0.5, s=35, color='#1f77b4', edgecolor='white', linewidth=0.4,
           label='Monthly Observations')
ax.plot(x_range * 100, y_fit * 100, color='#d62728', lw=2.2,
        label=f'OLS Fit: α = {alpha_annualized:.2%} (ann.), β = {beta:.2f}')

ax.axhline(0, color='black', lw=0.6, alpha=0.5)
ax.axvline(0, color='black', lw=0.6, alpha=0.5)
ax.set_xlabel('Market Excess Return (%)')
ax.set_ylabel('Strategy Excess Return (%)')
ax.set_title('CAPM Regression: Rₚ − Rₑ = α + β(Rₘ − Rₑ) + ε')

verdict = 'Significant at 5%' if p_value_alpha < 0.05 else 'NOT significant at 5%'
stats_text = (
    f"α (annualized): {alpha_annualized:+.2%}\n"
    f"β: {beta:.2f}\n"
    f"t-stat (α): {t_stat_alpha:.2f}\n"
    f"p-value (α): {p_value_alpha:.3f}\n"
    f"{verdict}"
)
ax.text(0.03, 0.97, stats_text, transform=ax.transAxes, fontsize=9.5,
        verticalalignment='top', horizontalalignment='left',
        bbox=dict(boxstyle='round', facecolor='white', edgecolor='gray', alpha=0.9))
ax.legend(loc='lower right', frameon=False, fontsize=9)
plt.tight_layout()
plt.savefig('step4_capm_scatter.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Alpha (annualized): {alpha_annualized:.2%} | t-stat: {t_stat_alpha:.2f} | p-value: {p_value_alpha:.4f}")
print("DECISION:", "Reject H0 — significant alpha." if p_value_alpha < 0.05 else "Fail to reject H0 — alpha not statistically distinguishable from zero.")

### 6. Rolling Alpha (with Rolling Beta for context)

Complements the static regression above: tracks whether the point estimate of alpha is consistently positive/negative through time, or whether the full-sample number is an average of very different sub-periods — directly relevant to interpreting the "is it significant" question.

In [ ]:
rolling_alpha = []
rolling_beta = []
rolling_se = []
rolling_dates = []

for i in range(ROLL_WINDOW, len(df) + 1):
    window = df.iloc[i - ROLL_WINDOW:i]
    X = sm.add_constant(window['Market_Excess'])
    yv = window['Strategy_Excess']
    m = sm.OLS(yv, X).fit()
    rolling_alpha.append(m.params['const'] * ANN_FACTOR)
    rolling_beta.append(m.params['Market_Excess'])
    rolling_se.append(m.bse['const'] * ANN_FACTOR)
    rolling_dates.append(window['rebalance_date'].iloc[-1])

roll_ab = pd.DataFrame({
    'rebalance_date': rolling_dates,
    'rolling_alpha': rolling_alpha,
    'rolling_beta': rolling_beta,
    'rolling_alpha_se': rolling_se,
})

fig, axes = plt.subplots(2, 1, figsize=(12, 8.5), sharex=True)

axes[0].plot(roll_ab['rebalance_date'], roll_ab['rolling_alpha'] * 100, color='#2ca02c', lw=1.8, label='Rolling Alpha (ann.)')
axes[0].fill_between(roll_ab['rebalance_date'],
                      (roll_ab['rolling_alpha'] - roll_ab['rolling_alpha_se']) * 100,
                      (roll_ab['rolling_alpha'] + roll_ab['rolling_alpha_se']) * 100,
                      color='#2ca02c', alpha=0.15, label='±1 Std. Error')
axes[0].axhline(0, color='black', lw=0.8, alpha=0.6)
axes[0].axhline(alpha_annualized, color='#d62728', lw=1.2, ls='--', label=f'Full-Sample Alpha ({alpha_annualized:.2%})')
axes[0].set_ylabel('Rolling Alpha (%, ann.)')
axes[0].set_title(f'Rolling {ROLL_WINDOW}-Month CAPM Alpha & Beta: Long-Short Spread vs. Market')
axes[0].legend(loc='best', frameon=False, fontsize=9)

axes[1].plot(roll_ab['rebalance_date'], roll_ab['rolling_beta'], color='#9467bd', lw=1.8, label='Rolling Beta')
axes[1].axhline(0, color='black', lw=0.8, alpha=0.6)
axes[1].axhline(beta, color='#d62728', lw=1.2, ls='--', label=f'Full-Sample Beta ({beta:.2f})')
axes[1].set_ylabel('Rolling Beta')
axes[1].set_xlabel('Date')
axes[1].legend(loc='best', frameon=False, fontsize=9)
axes[1].xaxis.set_major_locator(mdates.YearLocator(2))
axes[1].xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
fig.autofmt_xdate()

plt.tight_layout()
plt.savefig('step4_rolling_alpha_beta.png', dpi=150, bbox_inches='tight')
plt.show()

## Summary

Six charts saved, mapped one-to-one to the assignment's two remaining numbered steps:

**Step 3 (rolling backtest vs. benchmark):**
- `step3_equity_curve_vs_benchmark.png`
- `step3_rolling_metrics_panel.png`
- `step3_drawdown_comparison.png`
- `step3_summary_metrics_bar.png`

**Step 4 (alpha significance test):**
- `step4_capm_scatter.png`
- `step4_rolling_alpha_beta.png`